# 2.0.0 Kidney Replacement Calculator Demonstration

In version 2.0.0 of the dashboard stats calculator the algorithms to calculate the incident and prevalent cohorts have been fully overhauled. 

In [1]:
from rr_connection_manager import PostgresConnection

conn = PostgresConnection(app = "ukrdc_staging", tunnel = True, via_app = True)
conn.connection_check()
session = conn.session()


Connection to ukrdc_staging successful. 
Database info: 
	PostgreSQL 9.6.24 on x86_64-pc-linux-gnu


# Running Calculator 

The basic syntax is the same with all the calculators. The KRT calculator has a period of time implicit for the calculations of the incident cohort. For the backtesting this was set to a year to allow granular comparison with the cohorts in the annual report.

In [2]:
import json
import datetime as dt
from ukrdc_stats.calculators.krt import KRTStatsCalculator


facility = "RFBAK"
#start = dt.datetime(2021, 12, 31)
#end = dt.datetime(2022, 12, 31)

start = dt.datetime(2024, 1, 1)
end = dt.datetime.now()

calculator = KRTStatsCalculator(session=session, facility=facility, from_time=start, to_time=end)
output = calculator.extract_stats()



formatted_output = json.dumps(
    json.loads(
        output.all.json()
    ), 
    indent=4
)

print(formatted_output)


{
    "all_treatments_krt": {
        "metadata": {
            "title": "All KRT Modalities",
            "summary": "Breakdown of all patients on both PD and HD, and by home therapies and in-centre therapies.",
            "description": "\n# All Patients Undergoing Kidney Replacement Therapy\n\n## Overview\nThis pie chart illustrates the proportion of patients who received kidney replacement therapy within the time period. The chart is broken down by the type of treatment, including HD In-center, HD Home, HD Unknown/Incomplete, PD, and Tx. Optionally the chart can be filtered by satellite unit. \n\n## Treatment Definitions\n- HD: Haemodialysis patients (with a modality defined as HD by the UKRDC). This includes patients registered for haemodialysis, haemofiltration, haemodiafiltration, or ultrafiltration. \n- PD: Peritoneal dialysis (with a modality defined as PD by the UKRDC).This includes patients registered for CAPD or APD treatments.\n- TX: Transplant patients (with a modality d

In [6]:
from IPython.display import display, Markdown
import plotly.express as px


display(Markdown(output.all.prevalent_krt.metadata.description))
units = ", ".join(output.units.keys())

display(
    Markdown(
f"""
### Total Patient Population : {output.all.prevalent_krt.metadata.population_size}
### Satellite Units : {units}
"""
    )
)



all_patients = px.pie(
    names = output.all.prevalent_krt.data.x,
    values = output.all.prevalent_krt.data.y,
    hole=0.6,
)
all_patients.show()
#all_patients.show("png")




# Prevalent Patients Undergoing Kidney Replacement Therapy

## Overview
This pie chart illustrates the proportion of prevalent (to the end of the time window) patients who received kidney replacement therapy at a specified unit during a three-month period prior to the current date. The chart is broken down by type of treatment, including HD In-center, HD Home, HD Unknown/Incomplete, and PD. Optionally the chart can be filtered by satellite unit.

## Treatment Definitions
- HD: Haemodialysis patients (with a modality defined as HD by the UKRDC). This includes patients registered for haemodialysis, haemofiltration, haemodiafiltration, or ultrafiltration. 
- PD: Peritoneal dialysis (with a modality defined as PD by the UKRDC).This includes patients registered for CAPD or APD treatments.
- TX: Transplant patients (with a modality defined as TX), including both living and cadaver donors.
- In-centre: HD patients with qbl05 field of the Treatment table as HOSP or SATL.   
- Home: HD patients with qbl05 field of the Treatment table as HOME. 
- Unknown/Incomplete: HD patients with incomplete qbl05 field or anything other than HOME, HOSP, or SATL

## Study Methods
- The cohort was created from all patients admitted for HD or PD (as defined by the modality code mappings) at the specified unit or satellite unit.
- Any patients with a time of death before the beginning of the time window were excluded from the cohort, as were any patients whose treatments started before and ended after it.
- Any patient with a treatment to time or date of death before todays date are excluded
- Any patient with a transplant or dialysis treatment prior to the beginning of the time window was excluded.
- The numbers were calculated from the Patient and Treatment records in the UKRDC.
- Patient's therapy types was selected using the admission reason and the unit, and were further split into home and in-center therapy groups (with all patients on PD included in the home therapies group).
- Where there are multiple treatment modalities which overlap with the end of the time window the one with the most recent end date is selected. 

## UKRDC Entities Used
The chart was produced by joining the following UKRDC entities according to their foreign key relationships:
- [PatientRecord](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450149/PatientRecord): ukrdcid, sendingextract
- [Patient](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450145/Patient): deathtime
- [Treatment](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450155/Treatment+Encounter): qbl05, hdp04, fromtime, totime, dischargereasoncode, healthcarefacilitycode
- [ModalityCodes](https://renalregistry.atlassian.net/l/cp/Ac1YeFfH): registry_code_type



### Total Patient Population : 1314
### Satellite Units : RNQ51, RFBAT, RGN, RT5DC, 9RWDLB, RKZDA, RFBAK, RY5K7, 98RFBAK, P7U2H, 9RP7LA, 9RWD, 995


In [8]:
display(Markdown(output.all.incentre_dialysis_frequency.metadata.description))
freq_fig = px.bar(
    x=output.all.incentre_dialysis_frequency.data.y,
    y=output.all.incentre_dialysis_frequency.data.x,
    title=output.all.incentre_dialysis_frequency.metadata.title,
    labels={
        "x": output.all.incentre_dialysis_frequency.metadata.axis_titles.y,
        "y": output.all.incentre_dialysis_frequency.metadata.axis_titles.x,
    },
    orientation="h",
    color_discrete_sequence=["rgb(243,159,33)"],
    text_auto=True,
)
freq_fig.show()


# In-Centre Dialysis Frequency

## Overview
This histogram represents the mean number of dialysis sessions per week for all dialysis patients in a three month period at a sendingfacility or one of its satellites. Optionally the chart can be filtered by satellite unit. 

## Methodology
- Dialysis sessions are counted for patients in the 'All Patients Undergoing Kidney Replacement Therapy' cohort. This is done by grouping on the procedure type code. 
- Patients with less than two sessions are rejected. 
- The per week frequency is calculated for each person by dividing the count by the time difference between their first and last dialysis session within the three month period.
- Patients are aggregated into bins of with boundaries (0.5, 1.5, 2.5, 3.5, 7.0). This are labelled 1,2,3 and >3 sessions per week.  

## UKRDC Entities Used
The dialysis sessions table is queried by grouping by ukrdcid with the following aggregate functions used:
- https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2005565449/Dialysis+Session+Procedure: MIN(fromtime), MAX(totime), COUNT(sessiontype).


In [9]:
display(Markdown(output.all.incident_krt.metadata.description))

display(
    Markdown(
f"""
### Total Patient Population : {output.all.incident_krt.metadata.population_size}
"""
    )
)



all_patients = px.pie(
    names = output.all.incident_krt.data.x,
    values = output.all.incident_krt.data.y,
    hole=0.6,
)
all_patients.show()


# Incident Patients Undergoing Kidney Replacement Therapy

## Overview
This pie chart illustrates the modality of incident (new) kidney replacement therapy patients within the time window. The chart is broken down by type of treatment, including HD In-center, HD Home, HD Unknown/Incomplete, and PD. Optionally the chart can be filtered by satellite unit.

## Treatment Definitions
- HD: Haemodialysis patients (with a modality defined as HD by the UKRDC). This includes patients registered for haemodialysis, haemofiltration, haemodiafiltration, or ultrafiltration. 
- PD: Peritoneal dialysis (with a modality defined as PD by the UKRDC).This includes patients registered for CAPD or APD treatments.
- TX: Transplant patients (with a modality defined as TX), including both living and cadaver donors.
- In-centre: HD patients with qbl05 field of the Treatment table as HOSP or SATL.   
- Home: HD patients with qbl05 field of the Treatment table as HOME. 
- Unknown/Incomplete: HD patients with incomplete qbl05 field or anything other than HOME, HOSP, or SATL

## Study Methods
- The cohort was created from all patients admitted for kidney replacement therapy (as defined by the modality code mappings) at the specified unit or satellite unit.
- Any patients with a time of death before the beginning of the time window were excluded from the cohort, as were any patients whose treatments started before and ended after it.
- Any patient with a transplant or dialysis treatment prior to the beginning of the time window was excluded.
- The numbers were calculated from the Patient and Treatment records in the UKRDC.
- Patient's therapy types was selected using the admission reason and the unit, and were further split into home and in-center therapy groups (with all patients on PD included in the home therapies group).
- Where patients have multiple treatment records within the time window they are deduplicated using the treatment modality with the earliest starting date.

## UKRDC Entities Used
The chart was produced by joining the following UKRDC entities according to their foreign key relationships:
- [PatientRecord](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450149/PatientRecord): ukrdcid, sendingextract
- [Patient](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450145/Patient): deathtime
- [Treatment](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450155/Treatment+Encounter): qbl05, hdp04, fromtime, totime, dischargereasoncode, healthcarefacilitycode
- [ModalityCodes](https://renalregistry.atlassian.net/l/cp/Ac1YeFfH): registry_code_type



### Total Patient Population : 451
